## This demo showcases the implementation of user stories 404 and 582

This notebook shows the implementation of the different wrappers (rs-client-libraries) functions for the staging endpoints.

In [ ]:
import requests
import os
import pprint
import time
import pystac
from pystac import Asset, Collection, Extent, Item, SpatialExtent, TemporalExtent, ItemCollection
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"    
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10
collection_description = Collection(
    id=TEST_COLLECTION,
    description=None,  # rs-client will provide a default description for us
    extent=Extent(
        spatial=SpatialExtent(bboxes=[-180.0, -90.0, 180.0, 90.0]),
        temporal=TemporalExtent([start_date, stop_date]),
    ),
)

# Init the dask cluster
from resources.dask_utils import *
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.dask_utils import *

# Use the staging cluster
dask_gateway = dask_gateway_staging
dask_cluster = dask_cluster_staging

### Create a test collection in catalog named my_test_collection and check it afterwards to see if it's empty

In [ ]:
# Create a test collection 
collection = create_test_collection()

# Check the catalog for my_test_collection
items = catalog_client.get_items(TEST_COLLECTION)
assert not list(items)

### Check the available stac collections for stations (cadip, adgs) and for the stac catalog (the newly created one)

In [ ]:
for client in [auxip_client, cadip_client, catalog_client]:
    collections = client.get_collections()
    print(f"\nCollections response:")
    for collection in collections:
        print(f"ID: {collection.id}, Title: {collection.title}")
    

### Call the queryables (by using the rs-client) for .....

#### ADGS

In [ ]:
general_queryables = auxip_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)
print("\n\n")
collection_queryables = auxip_client.get_collection_queryables("adgs_aux_pp2")
assert isinstance(collection_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)

#### CADIP

In [ ]:
general_queryables = cadip_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)
print("\n\n")
collection_queryables = cadip_client.get_collection_queryables(cadip_collection_id)
assert isinstance(collection_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)

#### STAC Catalog

In [ ]:
general_queryables = catalog_client.get_queryables()
assert isinstance(general_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)
print("\n\n")
collection_queryables = catalog_client.get_collection_queryables(TEST_COLLECTION)
assert isinstance(collection_queryables, dict)
pprint.PrettyPrinter(indent=4).pprint(general_queryables)

### Get all the items from the collection "cadip_sentinel1" found in the configuration of the CADIP station
These items are in fact sessions in case of a cadip station
"get_items(<collection_name>) function should retrieve all the items from that collections


In [ ]:
items_collection_cadip = list(cadip_client.get_items(cadip_collection_id))
assert len(items_collection_cadip) > 0
for item in items_collection_cadip:
    print(f"Session {item.id} has {len(item.assets)} assets with datetime {item.properties.get('datetime')}")

### Request 14 items from the collection "adgs" found in the configuration of the ADGS station
By using the "search" function with a max_items set to 14, we are going to limit the result

In [ ]:
# Another example on how to get the items, will just load all the results in memory
# items_collection_adgs = list(auxip_client.get_items(adgs_collection_id))
items_collection_adgs = auxip_client.search(max_items = 14, collections = [adgs_collection_id])
assert len(items_collection_adgs) == 14

#pprint.PrettyPrinter(indent=4).pprint(items_collection_adgs)
for item in items_collection_adgs:
    print(f"AUXIP asset {item.id} has datetime {item.properties.get('datetime')}")

### Call the search method by using a cql2-text filter

In [ ]:
auxip_client.search(method = "GET", stac_filter="processing:facility='FOS' AND product:type='AUX_PP2'")

### Check existing processes + display information about the staging process

In [ ]:
# Returns list of all available processes from config.
processes = staging_client.get_processes()
print(processes)

In [ ]:
# Should return info about the staging process.
process_info = staging_client.get_process("staging")
print(process_info)

### Check the jobs table

In [ ]:
jobs = staging_client.get_jobs()
if jobs.get("numberMatched") > 0:
    delete_jobs = True
    if cluster_mode == True:
        delete_jobs = input(f"There are {jobs.get('numberMatched')} jobs in the table. Do you want to delete them all (y/n)?").lower().strip() == 'y'
    if delete_jobs:
        print("Deleting all the jobs...")
        for job in jobs.get("jobs"):
            delete_response = staging_client.delete_job(job.get("jobID"))
        # Check that the jobs have been deleted
        jobs = staging_client.get_jobs()
        print(f"Existing jobs: {jobs}")

### Starting 2 staging processes, one from the CADIP station and one from the ADGS station
The staging process from the ADGS station is expected to fail because one of the assets contains an incorrect download link for the file.

In [ ]:
staging_resp_list = []
for items in [pystac.ItemCollection(list(items_collection_cadip)), pystac.ItemCollection(list(items_collection_adgs))]:
    staging_resp_list.append(staging_client.run_staging(items.to_dict(), TEST_COLLECTION))
    
timeout = 120
started_job_id_list = []

for resp in staging_resp_list:
    started_job_id_list.append(resp["jobID"])
    while timeout > 0:
        if "running" not in resp["status"]:
            break
        # TODO: to replace with the following commented line after the rs-server-staging update
        ###job_info = staging_client.get_job_info(resp["jobID"])
        job_info = staging_client.get_job_info(resp["jobID"])
        pprint.PrettyPrinter(indent=4).pprint(job_info)
        print("\n")
        if "successful" in job_info["status"]:
            print(" ----- Job COMPLETED \n")
            break
        if "failed" in job_info["status"]:
            print("-----Job FAILED \n")
            break
        time.sleep(2)
        timeout -= 2

### Check the catalog collection "my_test_collection" for all the items:

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")


### Check the jobs table

In [ ]:
# Check that each of the job previously launched are successful
for job_id in started_job_id_list:
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

### Delete the catalog collection and recreate it

In [ ]:
# Create a test collection 
collection = create_test_collection()
# Check the catalog for my_test_collection
items = catalog_client.get_items(TEST_COLLECTION)
assert not list(items)

### Stage some files from both cadip and auxip stations

In [ ]:
items_cadip = stage_test_objects(cadip_client, 18, objects_are_files = True)
assert items_cadip
assert len(items_cadip.items[0].get_assets()) == 18
items_auxip = stage_test_objects(auxip_client, 14, objects_are_files = True)
assert items_auxip
assert len(items_auxip.items) == 14

### Check the catalog collection "my_test_collection" for all the items:

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))

for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")


### Example on how the user may stage all the items from the adgs collections, by using rs-client
First, delete the catalog collection and recreate it

In [ ]:
# Create a test collection 
collection = create_test_collection()
# Check the catalog for my_test_collection
items = catalog_client.get_items(TEST_COLLECTION)
assert not list(items)

### Stage all the items from the adgs collection that exists in the adgs station

In [ ]:
# Get all the items from the adgs colection
items_iterator = auxip_client.get_items("adgs")
# Create a pystac.ItemCollection from the result
item_collection = pystac.ItemCollection(list(items_iterator))

item_collection.items = [
    item for item in item_collection.items
    if item.properties.get("product:type") != "AX___OSF_AX"
]
# Use the dictionary to start the staging
job = staging_client.run_staging(item_collection.to_dict(), TEST_COLLECTION)
while timeout > 0:
    if "running" not in job["status"]:
        break
    # TODO: to replace with the following commented line after the rs-server-staging update
    ###job_info = staging_client.get_job_info(resp["jobID"])
    job_info = staging_client.get_job_info(job["jobID"])
    pprint.PrettyPrinter(indent=4).pprint(job_info)
    print("\n")
    if "successful" in job_info["status"]:
        print(" ----- Job COMPLETED \n")
        break
    if "failed" in job_info["status"]:
        print("-----Job FAILED \n")
        break
    time.sleep(2)
    timeout -= 2

### Check the catalog collection "my_test_collection" for all the items:

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_items(TEST_COLLECTION))
assert len(result) == 93
for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")


In [ ]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())